# 9장 실습 ② — 모델이 어디를 보는가 (Grad-CAM)

**PyTorch 판**

1장 §1.4에서 딥러닝이 못하는 일로 *"왜 그런지 설명하기"*를 꼽았습니다.
완전한 해법은 없지만, **어디를 보고 있는지**는 들여다볼 수 있습니다.

## 9.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot
from dlbook.data import Split

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 9.1 원천 과제와 목표 과제

**시험 800개·검증 400개를 고정**하고 학습 데이터만 줄입니다.
초고에서는 통째로 줄였다가 시험 12개짜리 표를 만들 뻔했습니다. (7장 §7.6)

In [ ]:
# 원천 과제 — 원·사각형·삼각형 6,000개
xs, ys = data.shapes(6000, seed=42, classes=(0, 1, 2))
src = data.split(xs, ys, val_ratio=0.15, test_ratio=0.15, seed=42)
print("원천 과제:", src)

# 목표 과제 — 십자·마름모. **원천 과제에 없던 도형이다.**
xt, yt = data.shapes(3000, seed=7, classes=(3, 4))

# ★ 시험 800개·검증 400개를 **고정**하고 학습 데이터만 줄인다.
#   (7장 §7.6 — 시험 데이터가 작으면 숫자를 믿을 수 없다.
#    초고에서 통째로 줄였다가 시험 12개짜리 표를 만들 뻔했다.)
x_test, y_test = xt[:800], yt[:800]
x_val, y_val = xt[800:1200], yt[800:1200]
pool_x, pool_y = xt[1200:], yt[1200:]

def target(n):
    """학습 데이터 n개짜리 목표 과제. 시험·검증은 언제나 같다."""
    return Split(pool_x[:n], pool_y[:n], x_val, y_val, x_test, y_test)

fig = plot.image_grid(np.concatenate([xs[:8], xt[:8]]),
                      np.concatenate([ys[:8], yt[:8] + 3]), n=16, cols=8,
                      class_names=list(data.SHAPE_CLASSES))
plt.show()
print("위: 원천 과제의 도형 / 아래: 목표 과제의 도형")

## 9.2 Grad-CAM — 여기만 판마다 다릅니다

**2장에서 배운 미분이 여기서도 쓰입니다.** 다만 이번에는
*"어느 쪽으로 얼마나 고칠까"* 가 아니라 *"어느 특징이 얼마나 중요했나"* 를
알아내는 데 씁니다.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
model_acc = 0.0

def to_nchw(a):
    return torch.tensor(np.asarray(a), dtype=torch.float32).permute(0, 3, 1, 2)

class SmallCNN(nn.Module):
    """마지막 합성곱의 출력을 따로 돌려주는 CNN. Grad-CAM에 필요하다."""
    def __init__(self, n_classes=3):
        super().__init__()
        self.c1 = nn.Conv2d(1, 16, 3, padding=1)
        self.c2 = nn.Conv2d(16, 32, 3, padding=1)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(32 * 7 * 7, 64),
                                  nn.ReLU(), nn.Linear(64, n_classes))

    def forward(self, x, return_conv=False):
        h = nn.functional.max_pool2d(torch.relu(self.c1(x)), 2)
        conv = torch.relu(self.c2(h))              # ← 마지막 합성곱
        out = self.head(nn.functional.max_pool2d(conv, 2))
        return (out, conv) if return_conv else out

def train_gradcam_model(sp):
    global model_acc, _model
    dlbook.set_seed(42)
    _model = SmallCNN(3).to(device)
    crit, opt = nn.CrossEntropyLoss(), torch.optim.Adam(_model.parameters(), 0.001)
    dl = DataLoader(TensorDataset(to_nchw(sp.x_train),
                                  torch.tensor(sp.y_train, dtype=torch.long)),
                    batch_size=64, shuffle=True)
    for _ in range(dlbook.smoke.epochs(15)):
        _model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); crit(_model(xb), yb).backward(); opt.step()
    _model.eval()
    with torch.no_grad():
        pred = _model(to_nchw(sp.x_test).to(device)).argmax(1).cpu().numpy()
    model_acc = metrics.accuracy(sp.y_test, pred)
    return _model

def gradcam(img):
    """마지막 합성곱 층의 특징 맵을, 판정에 기여한 정도로 가중 평균한다."""
    _model.eval()
    x = to_nchw(img[None]).to(device)
    out, conv = _model(x, return_conv=True)
    conv.retain_grad()
    cls = int(out[0].argmax())
    _model.zero_grad()
    out[0, cls].backward()                          # 2장의 그 미분이다
    weights = conv.grad[0].mean(dim=(1, 2))         # 특징 맵별 기여도
    cam = (conv[0].detach() * weights[:, None, None]).sum(0).cpu().numpy()
    cam = np.maximum(cam, 0)
    return cam / (cam.max() + 1e-8), cls

def upscale(cam, size):
    """작은 Grad-CAM 맵을 원본 크기로 늘린다. scipy 없이 numpy만 쓴다."""
    k = size // cam.shape[0]
    return np.kron(cam, np.ones((k, k)))[:size, :size]

## 9.3 그려 봅니다

In [ ]:
model = train_gradcam_model(src)
dlbook.record("ch09_gradcam_model_acc", model_acc)
print(f"도형 분류 CNN 정확도 {model_acc:.3f}")

names = data.shape_names((0, 1, 2))
sel = [np.flatnonzero(src.y_test == c)[k] for c in range(3) for k in (0, 1)]

fig, axes = plt.subplots(2, 6, figsize=(12.2, 4.6),
                         gridspec_kw={"hspace": 0.32, "wspace": 0.06})
for j, i in enumerate(sel):
    img = src.x_test[i]
    cam, cls = gradcam(img)
    axes[0, j].imshow(img.squeeze(), cmap="gray")
    axes[0, j].set_title(names[int(src.y_test[i])], fontsize=10.5, pad=5)
    axes[1, j].imshow(img.squeeze(), cmap="gray")
    axes[1, j].imshow(upscale(cam, img.shape[0]), cmap="jet", alpha=0.45)
    axes[1, j].set_title(f"→ {names[cls]}", fontsize=10.5, pad=5)
    for r in (0, 1):
        axes[r, j].set_xticks([]); axes[r, j].set_yticks([])
axes[0, 0].set_ylabel("원본"); axes[1, 0].set_ylabel("Grad-CAM")
plt.show()

print("→ 사각형과 삼각형은 **안쪽이 아니라 테두리**를 보고 있습니다.")
print("→ 당연합니다. 안쪽은 다 흰색이고, 구별되는 것은 모서리 모양뿐입니다.")
print("→ 모델이 사람과 같은 곳을 보고 있었습니다.")

## 정리

- **사각형과 삼각형은 안쪽이 아니라 테두리를 보고 있습니다.**
  안쪽은 다 흰색이고, 구별되는 것은 모서리 모양뿐입니다.
- Grad-CAM이 진짜 쓸모 있는 순간은 **모델이 엉뚱한 곳을 볼 때**입니다.
  성능이 이상하게 좋으면 의심하고 확인하십시오.

### 연습

1. **틀린 예측**을 골라 Grad-CAM을 그리십시오. 맞힌 것과 무엇이 다릅니까.
2. 도형에 **일부러 편향을 심으십시오.** "삼각형" 영상에만 왼쪽 위에 밝은
   점을 찍습니다. 정확도는 얼마가 나오고, Grad-CAM은 어디를 봅니까.
3. 그 점이 없는 삼각형 영상에서는 어떻게 됩니까. (6장 §6.6 분포 이동)